## Data Preprocessing

This section audits dataset quality, removes near-duplicates, checks class balance, verifies train/val/test splits, applies letterbox resizing to 640x640, and augments training images — all before model training begins.

In [ ]:
%pip install imagehash albumentations -q

from pathlib import Path
from PIL import Image
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import cv2

# ── Configuration ─────────────────────────────────────────────────────────────
DATASET_PATH = Path('../data')
CLASSES = ['Sunglasses', 'Keys', 'Wallet']
SPLITS = ['train', 'valid', 'test']
IMG_SIZE = 640

### Step 1: Dataset Quality Audit

Count images per split and class, detect near-duplicates via perceptual hashing, identify corrupted images, and flag missing or empty annotation files.

In [ ]:
import imagehash

print('=' * 60)
print('STEP 1: Dataset Quality Audit')
print('=' * 60)

# 1a. Image counts per split
print('\n--- Images per split ---')
for split in SPLITS:
    imgs = [f for f in (DATASET_PATH / split / 'images').glob('*')
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    print(f'  {split}: {len(imgs)} images')

# 1b. Instance counts per class per split
print('\n--- Instances per class per split ---')
for split in SPLITS:
    counts = Counter()
    for lbl in (DATASET_PATH / split / 'labels').glob('*.txt'):
        for line in lbl.read_text().strip().splitlines():
            if line.strip():
                cls_id = int(line.split()[0])
                if cls_id < len(CLASSES):
                    counts[CLASSES[cls_id]] += 1
    print(f'  {split}: {dict(counts)}')

# 1c. Near-duplicate detection via perceptual hashing
print('\n--- Near-duplicate detection ---')
hash_map = {}
duplicates = []
for split in SPLITS:
    for img_path in sorted((DATASET_PATH / split / 'images').glob('*')):
        if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue
        try:
            h = str(imagehash.phash(Image.open(img_path)))
            if h in hash_map:
                duplicates.append((hash_map[h], (split, img_path.name)))
            else:
                hash_map[h] = (split, img_path.name)
        except Exception:
            pass  # handled in corrupted check below

if duplicates:
    print(f'  Found {len(duplicates)} near-duplicate pair(s):')
    for orig, dup in duplicates[:10]:
        print(f'    {orig[0]}/{orig[1]}  <->  {dup[0]}/{dup[1]}')
    if len(duplicates) > 10:
        print(f'    ... and {len(duplicates) - 10} more')
else:
    print('  No near-duplicates detected.')

# 1d. Corrupted / unreadable images
print('\n--- Corrupted images ---')
corrupted = []
for split in SPLITS:
    for img_path in sorted((DATASET_PATH / split / 'images').glob('*')):
        if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue
        try:
            with Image.open(img_path) as im:
                im.verify()
        except Exception as e:
            corrupted.append(f'{split}/{img_path.name}: {e}')

if corrupted:
    print(f'  Found {len(corrupted)} corrupted image(s):')
    for c in corrupted:
        print(f'    {c}')
else:
    print('  All images readable.')

# 1e. Missing or empty annotation files
print('\n--- Missing / empty annotations ---')
missing, empty = [], []
for split in SPLITS:
    imgs_dir = DATASET_PATH / split / 'images'
    labels_dir = DATASET_PATH / split / 'labels'
    for img_path in sorted(imgs_dir.glob('*')):
        if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue
        lbl = labels_dir / (img_path.stem + '.txt')
        if not lbl.exists():
            missing.append(f'{split}/{img_path.name}')
        elif not lbl.read_text().strip():
            empty.append(f'{split}/{img_path.name}')

print(f'  Missing label files: {len(missing)}')
for m in missing[:5]:
    print(f'    {m}')
if len(missing) > 5:
    print(f'    ... and {len(missing) - 5} more')
print(f'  Empty label files (background / negative samples): {len(empty)}')
for e in empty[:5]:
    print(f'    {e}')
if len(empty) > 5:
    print(f'    ... and {len(empty) - 5} more')

print('\nStep 1 complete.')

### Step 2: Near-Duplicate Removal

Remove one image from each near-duplicate pair detected in Step 1. This prevents the model from memorising repeated examples and ensures cleaner evaluation metrics. The **second** occurrence (the duplicate) is deleted along with its label file.

In [ ]:
print('=' * 60)
print('STEP 2: Near-Duplicate Removal')
print('=' * 60)

removed = 0
kept_empty_removed = 0

for orig, dup in duplicates:
    dup_split, dup_name = dup
    img_path = DATASET_PATH / dup_split / 'images' / dup_name
    lbl_path = DATASET_PATH / dup_split / 'labels' / (Path(dup_name).stem + '.txt')

    if img_path.exists():
        img_path.unlink()
        removed += 1
    if lbl_path.exists():
        # Check if the label was empty (background sample)
        if not lbl_path.read_text().strip():
            kept_empty_removed += 1
        lbl_path.unlink()

print(f'  Removed {removed} duplicate images (and their labels).')
print(f'  Of those, {kept_empty_removed} had empty labels (background samples).')

# Recount after removal
print('\n--- Post-dedup image counts ---')
for split in SPLITS:
    imgs = [f for f in (DATASET_PATH / split / 'images').glob('*')
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    print(f'  {split}: {len(imgs)} images')

print('\n--- Post-dedup instances per class per split ---')
for split in SPLITS:
    counts = Counter()
    for lbl in (DATASET_PATH / split / 'labels').glob('*.txt'):
        for line in lbl.read_text().strip().splitlines():
            if line.strip():
                cls_id = int(line.split()[0])
                if cls_id < len(CLASSES):
                    counts[CLASSES[cls_id]] += 1
    print(f'  {split}: {dict(counts)}')

print('\nStep 2 complete.')

### Step 3: Class Balance Check

Count bounding box instances per class across the full dataset. Flag any class whose count falls below 70% of the most-frequent class.

In [ ]:
# Count bounding box instances per class across all splits
total_counts = Counter()
for split in SPLITS:
    for lbl in (DATASET_PATH / split / 'labels').glob('*.txt'):
        for line in lbl.read_text().strip().splitlines():
            if line.strip():
                cls_id = int(line.split()[0])
                if cls_id < len(CLASSES):
                    total_counts[CLASSES[cls_id]] += 1

# Plot class distribution
counts_list = [total_counts.get(cls, 0) for cls in CLASSES]
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(CLASSES, counts_list, color=['#4e79a7', '#f28e2b', '#e15759'])
ax.set_ylabel('Bounding Box Instances')
ax.set_title('Class Distribution (All Splits Combined)')
for bar, count in zip(bars, counts_list):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            str(count), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

# Flag under-represented classes (< 70% of the most frequent)
max_count = max(counts_list)
threshold = 0.7 * max_count
print('Class balance check (threshold: 70% of most frequent):')
for cls, count in zip(CLASSES, counts_list):
    status = 'OK' if count >= threshold else f'LOW (< {threshold:.0f})'
    print(f'  {cls}: {count} — {status}')

print('\nStep 2 complete.')

### Step 4: Train / Val / Test Split Verification

Confirm the existing Roboflow splits are close to the standard 70/20/10 or 80/10/10 ratio. Warn if any split deviates by more than 5 percentage points.

In [ ]:
# Count images per split
split_counts = {}
for split in SPLITS:
    imgs = [f for f in (DATASET_PATH / split / 'images').glob('*')
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    split_counts[split] = len(imgs)

total = sum(split_counts.values())
ratios = {s: c / total * 100 for s, c in split_counts.items()}

print('Split ratios:')
for split, count in split_counts.items():
    print(f'  {split:6s}: {count:5d} images ({ratios[split]:5.1f}%)')
print(f'  {"total":6s}: {total:5d} images')

# Check against standard targets
targets = [
    {'name': '70/10/20', 'train': 70, 'valid': 10, 'test': 20},
    {'name': '80/10/10', 'train': 80, 'valid': 10, 'test': 10},
]

print('\nDeviation from standard splits:')
for t in targets:
    deviations = {s: abs(ratios[s] - t[s]) for s in SPLITS}
    max_dev = max(deviations.values())
    status = 'OK' if max_dev < 5 else 'WARNING — significant deviation'
    print(f'  vs {t["name"]}: max deviation = {max_dev:.1f}% — {status}')

print('\nStep 3 complete.')

### Step 5: Image Resizing with Letterboxing

Resize all images to 640x640 using aspect-preserving letterbox padding (grey fill). YOLO bounding box annotations are rescaled accordingly.

In [ ]:
def letterbox_resize(img, target=640, pad_color=(114, 114, 114)):
    """Resize image to target x target with grey letterbox padding.
    Returns the padded image, scale factor, and x/y offsets."""
    h, w = img.shape[:2]
    scale = min(target / w, target / h)
    nw, nh = int(w * scale), int(h * scale)

    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((target, target, 3), pad_color, dtype=np.uint8)
    x_off = (target - nw) // 2
    y_off = (target - nh) // 2
    canvas[y_off:y_off + nh, x_off:x_off + nw] = resized
    return canvas, scale, x_off, y_off

print('Resizing images to 640x640 with letterboxing...')
for split in SPLITS:
    imgs_dir = DATASET_PATH / split / 'images'
    labels_dir = DATASET_PATH / split / 'labels'
    resized_count, skipped = 0, 0

    for img_path in sorted(imgs_dir.glob('*')):
        if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        if w == IMG_SIZE and h == IMG_SIZE:
            skipped += 1
            continue

        padded, scale, x_off, y_off = letterbox_resize(img, IMG_SIZE)

        # Update YOLO annotations to match the letterbox transform
        lbl_path = labels_dir / (img_path.stem + '.txt')
        if lbl_path.exists() and lbl_path.read_text().strip():
            new_lines = []
            for line in lbl_path.read_text().strip().splitlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = parts[0]
                xc, yc, bw, bh = map(float, parts[1:5])

                # Denormalise -> apply scale + offset -> renormalise to 640
                xc = (xc * w * scale + x_off) / IMG_SIZE
                yc = (yc * h * scale + y_off) / IMG_SIZE
                bw = bw * w * scale / IMG_SIZE
                bh = bh * h * scale / IMG_SIZE
                new_lines.append(f'{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
            lbl_path.write_text('\n'.join(new_lines) + '\n')

        cv2.imwrite(str(img_path), padded)
        resized_count += 1

    print(f'  {split}: {resized_count} resized, {skipped} already 640x640')

print('\nStep 4 complete.')

### Step 6: Augmentation Pipeline

Apply data augmentations to the **training set only** to increase variety. Each training image gets 2 augmented copies with randomised brightness, flips, rotation, crop, blur, and colour jitter. Bounding boxes are transformed in sync.

In [ ]:
import albumentations as A

# Save pre-augmentation image count for the final summary
pre_aug_train_count = len([f for f in (DATASET_PATH / 'train' / 'images').glob('*')
                           if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])

# Augmentation pipeline with bounding box support
transform = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT,
             value=(114, 114, 114), p=0.5),
    A.RandomResizedCrop(
        size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0), ratio=(0.9, 1.1), p=0.5),
    A.MotionBlur(blur_limit=7, p=0.3),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30,
                         val_shift_limit=20, p=0.4),
], bbox_params=A.BboxParams(
    format='yolo', label_fields=['class_labels'], min_visibility=0.3))

NUM_AUG = 2  # augmented copies per original image
train_imgs = DATASET_PATH / 'train' / 'images'
train_lbls = DATASET_PATH / 'train' / 'labels'

# Only augment original images (skip any previously generated _aug files)
img_files = sorted([f for f in train_imgs.glob('*')
                    if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
                    and '_aug' not in f.stem])

print(f'Augmenting {len(img_files)} training images ({NUM_AUG} copies each)...')
aug_count = 0

for idx, img_path in enumerate(img_files):
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Read YOLO annotations
    lbl_path = train_lbls / (img_path.stem + '.txt')
    bboxes, class_labels = [], []
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                class_labels.append(int(parts[0]))
                bboxes.append([float(x) for x in parts[1:5]])

    if not bboxes:
        continue  # skip background images

    for i in range(NUM_AUG):
        try:
            result = transform(
                image=img_rgb, bboxes=bboxes, class_labels=class_labels)

            if not result['bboxes']:
                continue  # all boxes cropped out

            # Save augmented image
            aug_stem = f'{img_path.stem}_aug{i}'
            aug_bgr = cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR)
            cv2.imwrite(str(train_imgs / f'{aug_stem}{img_path.suffix}'), aug_bgr)

            # Save augmented labels
            lines = [f'{int(c)} {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}'
                     for c, b in zip(result['class_labels'], result['bboxes'])]
            (train_lbls / f'{aug_stem}.txt').write_text('\n'.join(lines) + '\n')

            aug_count += 1
        except Exception:
            continue

    if (idx + 1) % 500 == 0:
        print(f'  Processed {idx + 1}/{len(img_files)} images...')

print(f'\nGenerated {aug_count} augmented images.')
print(f'Training set: {pre_aug_train_count} -> {pre_aug_train_count + aug_count} images')
print('\nStep 5 complete.')


### Step 7: Final Dataset Summary

Summary of the dataset after all preprocessing steps: image counts, per-class instance counts, and a dimension spot-check to verify letterboxing.

In [ ]:
print('=' * 60)
print('FINAL DATASET SUMMARY')
print('=' * 60)

for split in SPLITS:
    imgs_dir = DATASET_PATH / split / 'images'
    labels_dir = DATASET_PATH / split / 'labels'

    img_files = [f for f in imgs_dir.glob('*')
                 if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]

    # Count class instances
    class_counts = Counter()
    for lbl in labels_dir.glob('*.txt'):
        for line in lbl.read_text().strip().splitlines():
            if line.strip():
                cls_id = int(float(line.split()[0]))
                if cls_id < len(CLASSES):
                    class_counts[CLASSES[cls_id]] += 1

    # Spot-check image dimensions (first 50)
    non_target = 0
    for f in img_files[:50]:
        with Image.open(f) as im:
            if im.size != (IMG_SIZE, IMG_SIZE):
                non_target += 1

    print(f'\n{split.upper()} ({len(img_files)} images):')
    if split == 'train':
        try:
            print(f'  Before augmentation: {pre_aug_train_count}')
            print(f'  After augmentation:  {len(img_files)}')
        except NameError:
            print(f'  Total: {len(img_files)}')
    for cls in CLASSES:
        print(f'  {cls}: {class_counts.get(cls, 0)} instances')
    if non_target == 0:
        print(f'  Dimensions: all {IMG_SIZE}x{IMG_SIZE} (spot-checked)')
    else:
        print(f'  WARNING: {non_target}/50 images not {IMG_SIZE}x{IMG_SIZE}')

print('\n' + '=' * 60)
print('Preprocessing complete. Ready for training.')
print('=' * 60)

## YOLO Model Training & Evaluation

Train all six YOLO variants (nano and small tiers across three architectures) on the preprocessed dataset, then evaluate each on the held-out test split. The best-performing variant will be selected as the YOLO representative in the final 3-architecture comparison.

| Architecture | Nano | Small |
|---|---|---|
| YOLOv8 | `yolov8n.pt` | `yolov8s.pt` |
| YOLO11 | `yolo11n.pt` | `yolo11s.pt` |
| YOLO26 | `yolo26n.pt` | `yolo26s.pt` |

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLOv8n model
model = YOLO('yolov8n.pt')

# Train the model - note the ../ to go up one directory level
results = model.train(
    data='../data/data.yaml',    # Go up one level, then into data folder
    epochs=50,
    imgsz=640,
    batch=16,
    name='yolov8n_run1',
    patience=10,
    device=0,
    workers=0
)

In [ ]:
from ultralytics import YOLO

# Load the best trained model
model = YOLO('runs/detect/yolov8n_run1/weights/best.pt')

# Evaluate on test set
metrics = model.val(data='../data/data.yaml', split='test')

# Print key metrics
print(f"mAP50: {metrics.box.map50}")
print(f"mAP50-95: {metrics.box.map}")
print(f"Precision: {metrics.box.mp}")
print(f"Recall: {metrics.box.mr}")

# Check per-class performance
print("\nPer-class mAP50:")
for i, name in enumerate(['Sunglasses', 'Keys', 'Wallet']):
    print(f"{name}: {metrics.box.maps[i]}")

In [ ]:
from ultralytics import YOLO

# Load YOLO26 nano model
model = YOLO('yolo26n.pt')

# Train
results = model.train(
    data='../data/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='yolo26n_run1',
    patience=10,
    device=0,
    workers=0
)

In [1]:
from ultralytics import YOLO

# Load the best trained YOLO26n model
model = YOLO('runs/detect/yolo26n_run1/weights/best.pt')

# Evaluate on test set
metrics = model.val(data='../data/data.yaml', split='test')

# Print key metrics
print(f"mAP50: {metrics.box.map50}")
print(f"mAP50-95: {metrics.box.map}")
print(f"Precision: {metrics.box.mp}")
print(f"Recall: {metrics.box.mr}")

# Check per-class performance
print("\nPer-class mAP50:")
for i, name in enumerate(['Sunglasses', 'Keys', 'Wallet']):
    print(f"{name}: {metrics.box.maps[i]}")

Ultralytics 8.4.14 🚀 Python-3.10.19 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
YOLO26n summary (fused): 122 layers, 2,375,421 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.2±0.2 ms, read: 250.3±80.5 MB/s, size: 59.7 KB)
val: Scanning /home/bradtab/git/FYP/ml-training/data/test/labels... 186 images, 10 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 186/186 1.6Kit/s 0.1s<0.1s
val: New cache created: /home/bradtab/git/FYP/ml-training/data/test/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 19, len(boxes) = 219. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
WARNING ⚠️ NMS time limit 2.050s exceeded
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 1.3s/it 15.0s0.3s
                   all        186        219      0.937     

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='../data/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='yolov8s_run1',
    patience=10,
    device=0,
    workers=0
)

In [2]:
from ultralytics import YOLO

# Load the best trained YOLOv8s model
model_v8s   = YOLO('runs/detect/yolov8s_run1/weights/best.pt')
metrics_v8s = model_v8s.val(data='../data/data.yaml', split='test')

print('YOLOv8s (50 epochs)')
print(f'  mAP50:     {metrics_v8s.box.map50:.4f}')
print(f'  mAP50-95:  {metrics_v8s.box.map:.4f}')
print(f'  Precision: {metrics_v8s.box.mp:.4f}')
print(f'  Recall:    {metrics_v8s.box.mr:.4f}')
print('\n  Per-class mAP50:')
for i, name in enumerate(['Sunglasses', 'Keys', 'Wallet']):
    print(f'    {name}: {metrics_v8s.box.maps[i]:.4f}')

Ultralytics 8.4.14 🚀 Python-3.10.19 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 342.6±206.8 MB/s, size: 70.3 KB)
val: Scanning /home/bradtab/git/FYP/ml-training/data/test/labels.cache... 186 images, 10 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 186/186 43.3Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 19, len(boxes) = 219. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 10.6it/s 1.1s0.1s
                   all        186        219      0.918      0.927      0.923      0.747
            Sunglasses         39         39       0.97      0.949      0.961      0.772
   

In [4]:
from ultralytics import YOLO

model = YOLO('yolo26s.pt')

results = model.train(
    data='../data/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='yolo26s_run1',
    patience=10,
    device=0,
    workers=0
)

New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.10.19 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26s_run1, 

/home/bradtab/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


val: Scanning /home/bradtab/git/FYP/ml-training/data/valid/labels... 181 images, 10 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 181/181 1.9Kit/s 0.1s
val: New cache created: /home/bradtab/git/FYP/ml-training/data/valid/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 22, len(boxes) = 217. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001429, momentum=0.9) with parameter groups 114 weight(decay=0.0), 126 weight(decay=0.0005), 126 bias(decay=0.0)
Plotting labels to /home/bradtab/git/FYP/ml-training/notebooks/runs/detect/yolo26s_run1/labels.jpg... 
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /home/bradtab/git/FYP/ml-training/n

In [5]:
from ultralytics import YOLO

# Load the best trained YOLO26s model
model_y26s   = YOLO('runs/detect/yolo26s_run1/weights/best.pt')
metrics_y26s = model_y26s.val(data='../data/data.yaml', split='test')

print('YOLO26s (50 epochs)')
print(f'  mAP50:     {metrics_y26s.box.map50:.4f}')
print(f'  mAP50-95:  {metrics_y26s.box.map:.4f}')
print(f'  Precision: {metrics_y26s.box.mp:.4f}')
print(f'  Recall:    {metrics_y26s.box.mr:.4f}')
print('\n  Per-class mAP50:')
for i, name in enumerate(['Sunglasses', 'Keys', 'Wallet']):
    print(f'    {name}: {metrics_y26s.box.maps[i]:.4f}')

Ultralytics 8.4.14 🚀 Python-3.10.19 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
YOLO26s summary (fused): 122 layers, 9,466,341 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 123.3±33.3 MB/s, size: 54.3 KB)
val: Scanning /home/bradtab/git/FYP/ml-training/data/test/labels.cache... 186 images, 10 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 186/186 1.9Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 19, len(boxes) = 219. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 4.4it/s 2.7s<0.2s
                   all        186        219      0.881      0.919      0.926      0.743
            Sunglasses         39         39      0.875      0.923      0.937      0.742
   

In [1]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data='../data/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='yolo11n_run1',
    patience=10,
    device=0,
    workers=0
)

New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.10.19 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_run1, 

/home/bradtab/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


val: Scanning /home/bradtab/git/FYP/ml-training/data/valid/labels... 181 images, 10 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 181/181 1.7Kit/s 0.1s<0.0s
val: New cache created: /home/bradtab/git/FYP/ml-training/data/valid/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 22, len(boxes) = 217. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001429, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Plotting labels to /home/bradtab/git/FYP/ml-training/notebooks/runs/detect/yolo11n_run1/labels.jpg... 
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /home/bradtab/git/FYP/ml-training

In [2]:
from ultralytics import YOLO

# Load the best trained YOLO11n model
model_y11n   = YOLO('runs/detect/yolo11n_run1/weights/best.pt')
metrics_y11n = model_y11n.val(data='../data/data.yaml', split='test')

print('YOLO11n (50 epochs)')
print(f'  mAP50:     {metrics_y11n.box.map50:.4f}')
print(f'  mAP50-95:  {metrics_y11n.box.map:.4f}')
print(f'  Precision: {metrics_y11n.box.mp:.4f}')
print(f'  Recall:    {metrics_y11n.box.mr:.4f}')
print('\n  Per-class mAP50:')
for i, name in enumerate(['Sunglasses', 'Keys', 'Wallet']):
    print(f'    {name}: {metrics_y11n.box.maps[i]:.4f}')

Ultralytics 8.4.14 🚀 Python-3.10.19 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 341.2±253.3 MB/s, size: 64.8 KB)
val: Scanning /home/bradtab/git/FYP/ml-training/data/test/labels... 186 images, 10 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 186/186 1.7Kit/s 0.1s<0.0s
val: New cache created: /home/bradtab/git/FYP/ml-training/data/test/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 19, len(boxes) = 219. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 9.2it/s 1.3s<0.2s
                   all        186        219      0.893      0.938       0.93      0.736
            

In [3]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

results = model.train(
    data='../data/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='yolo11s_run1',
    patience=10,
    device=0,
    workers=0
)

New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.10.19 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11s_run1, 

In [4]:
from ultralytics import YOLO

# Load the best trained YOLO11s model
model_y11s   = YOLO('runs/detect/yolo11s_run1/weights/best.pt')
metrics_y11s = model_y11s.val(data='../data/data.yaml', split='test')

print('YOLO11s (50 epochs)')
print(f'  mAP50:     {metrics_y11s.box.map50:.4f}')
print(f'  mAP50-95:  {metrics_y11s.box.map:.4f}')
print(f'  Precision: {metrics_y11s.box.mp:.4f}')
print(f'  Recall:    {metrics_y11s.box.mr:.4f}')
print('\n  Per-class mAP50:')
for i, name in enumerate(['Sunglasses', 'Keys', 'Wallet']):
    print(f'    {name}: {metrics_y11s.box.maps[i]:.4f}')

Ultralytics 8.4.14 🚀 Python-3.10.19 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
YOLO11s summary (fused): 101 layers, 9,413,961 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3220.4±839.7 MB/s, size: 54.3 KB)
val: Scanning /home/bradtab/git/FYP/ml-training/data/test/labels.cache... 186 images, 10 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 186/186 52.0Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 19, len(boxes) = 219. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 10.0it/s 1.2s0.2s
                   all        186        219      0.914      0.933      0.931      0.747
            Sunglasses         39         39      0.921      0.923      0.965      0.778


## RF-DETR Pipeline

RF-DETR is Roboflow's transformer-based detector. Two preparation steps are needed before training:

1. **Install** the `rfdetr` package
2. **Convert** the dataset from YOLO `.txt` format to COCO JSON (required by RF-DETR — it does not accept YOLO format directly)

> **Model choice:** RF-DETR Base (~29M parameters) is used as the smallest available variant. It is considerably larger than the YOLO nano models, making this comparison an explicit study of the **accuracy vs. efficiency trade-off** between lightweight CNN-based detectors and heavier transformer-based detectors — a core argument for your FYP.

In [ ]:
%pip install rfdetr -q

In [1]:
import json
from pathlib import Path
from PIL import Image

CLASSES = ['Sunglasses', 'Keys', 'Wallet']

def yolo_to_coco(images_dir, labels_dir, output_json, split_name=''):
    """Convert YOLO txt annotations to COCO JSON.

    YOLO format : class_id  x_center  y_center  width  height  (normalised 0–1)
    COCO format : [x_top_left_px, y_top_left_px, width_px, height_px]

    file_name is stored as 'images/<filename>' so that RF-DETR, which resolves
    paths as  dataset_dir / split / file_name,  finds the image at the correct
    location (dataset_dir / split / images / <filename>).
    """
    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)

    coco = {
        'images':      [],
        'annotations': [],
        # supercategory is required by RF-DETR; it filters out any entry where
        # supercategory == "none", so we use the class name as the supercategory.
        'categories':  [{'id': i, 'name': cls, 'supercategory': cls}
                        for i, cls in enumerate(CLASSES)]
    }
    ann_id, img_id = 0, 0

    img_paths = sorted(images_dir.glob('*.jpg')) + sorted(images_dir.glob('*.png'))
    for img_path in img_paths:
        with Image.open(img_path) as img:
            w, h = img.size

        coco['images'].append({
            'id':        img_id,
            'file_name': 'images/' + img_path.name,   # include subdirectory prefix
            'width':     w,
            'height':    h
        })

        label_path = labels_dir / (img_path.stem + '.txt')
        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    parts  = line.split()
                    cls_id = int(parts[0])
                    x_c, y_c, bw, bh = map(float, parts[1:5])

                    # Denormalise and convert centre-based → top-left origin
                    x_px  = (x_c - bw / 2) * w
                    y_px  = (y_c - bh / 2) * h
                    bw_px = bw * w
                    bh_px = bh * h

                    coco['annotations'].append({
                        'id': ann_id, 'image_id': img_id, 'category_id': cls_id,
                        'bbox': [x_px, y_px, bw_px, bh_px],
                        'area': bw_px * bh_px, 'iscrowd': 0
                    })
                    ann_id += 1
        img_id += 1

    with open(output_json, 'w') as f:
        json.dump(coco, f)
    print(f'  {split_name}: {len(coco["images"])} images, '
          f'{len(coco["annotations"])} annotations → {output_json}')

print('Converting YOLO annotations to COCO JSON...')
base = Path('../data')
for split_name, folder in [('train', 'train'), ('valid', 'valid'), ('test', 'test')]:
    yolo_to_coco(
        images_dir=base / folder / 'images',
        labels_dir=base / folder / 'labels',
        output_json=base / folder / '_annotations.coco.json',
        split_name=split_name
    )
print('Done.')

Converting YOLO annotations to COCO JSON...
  train: 4739 images, 5498 annotations → ../data/train/_annotations.coco.json
  valid: 181 images, 217 annotations → ../data/valid/_annotations.coco.json
  test: 186 images, 219 annotations → ../data/test/_annotations.coco.json
Done.


In [2]:
from rfdetr import RFDETRBase

# RF-DETR expects dataset_dir to contain train/, valid/, test/ subdirectories,
# each with image files and a _annotations.coco.json (generated in the cell above).
rfdetr_model = RFDETRBase()

rfdetr_model.train(
    dataset_dir='../data',
    epochs=50,
    batch_size=8,
    grad_accum_steps=4,   # effective batch size = 32
    lr=1e-4,
    output_dir='runs/rfdetr/rfdetr_base_run1'
)

2026-03-25 10:19:59.289410: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-25 10:19:59.459473: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774430399.529177    1947 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774430399.550023    1947 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774430399.718533    1947 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

[2026-03-25 10:20:02] [INFO] rf-detr - File rf-detr-base.pth already exists with correct MD5 hash.
[2026-03-25 10:20:02] [INFO] rf-detr - Loading pretrain weights


[2026-03-25 10:20:03] [WARNING] rf-detr - Reinitializing your detection head with 4 classes.


[2026-03-25 10:20:03] [INFO] rf-detr - TensorBoard logging initialized. To monitor logs, use 'tensorboard --logdir /home/bradtab/git/FYP/ml-training/notebooks/runs/rfdetr/rfdetr_base_run1' and open http://localhost:6006/ in browser.
[2026-03-25 10:20:03] [INFO] rf-detr - Not using distributed mode
[2026-03-25 10:20:03] [INFO] rf-detr - git:
  unknown

[2026-03-25 10:20:03] [INFO] rf-detr - Namespace(num_classes=4, grad_accum_steps=4, print_freq=10, amp=True, lr=0.0001, lr_encoder=0.00015, batch_size=8, weight_decay=0.0001, epochs=50, lr_drop=100, clip_max_norm=0.1, lr_vit_layer_decay=0.8, lr_component_decay=0.7, do_benchmark=False, dropout=0, drop_path=0.0, drop_mode='standard', drop_schedule='constant', cutoff_epoch=0, pretrained_encoder=None, pretrain_weights='rf-detr-base.pth', pretrain_exclude_keys=None, pretrain_keys_modify_to_load=None, pretrained_distiller=None, encoder='dinov2_windowed_small', vit_encoder_num_layers=12, window_block_indexes=None, position_embedding='sine', out_

fatal: not a git repository (or any of the parent directories): .git


[2026-03-25 10:20:06] [INFO] rf-detr - Epoch: [0]  [  0/148]  eta: 0:07:49  lr: 0.000100  class_error: 75.00  loss: 10.5862 (10.5862)  loss_ce: 1.3541 (1.3541)  loss_bbox: 0.8919 (0.8919)  loss_giou: 0.5581 (0.5581)  loss_ce_0: 1.2019 (1.2019)  loss_bbox_0: 0.8791 (0.8791)  loss_giou_0: 0.5751 (0.5751)  loss_ce_1: 1.2975 (1.2975)  loss_bbox_1: 0.9046 (0.9046)  loss_giou_1: 0.5441 (0.5441)  loss_ce_enc: 1.1860 (1.1860)  loss_bbox_enc: 0.6470 (0.6470)  loss_giou_enc: 0.5468 (0.5468)  loss_ce_unscaled: 1.3541 (1.3541)  class_error_unscaled: 75.0000 (75.0000)  loss_bbox_unscaled: 0.1784 (0.1784)  loss_giou_unscaled: 0.2791 (0.2791)  cardinality_error_unscaled: 3766.3750 (3766.3750)  loss_ce_0_unscaled: 1.2019 (1.2019)  loss_bbox_0_unscaled: 0.1758 (0.1758)  loss_giou_0_unscaled: 0.2876 (0.2876)  cardinality_error_0_unscaled: 3636.0000 (3636.0000)  loss_ce_1_unscaled: 1.2975 (1.2975)  loss_bbox_1_unscaled: 0.1809 (0.1809)  loss_giou_1_unscaled: 0.2720 (0.2720)  cardinality_error_1_unscaled:

In [3]:
import json, os, time
from pathlib import Path
from PIL import Image
from rfdetr import RFDETRBase

RESULTS_JSON    = 'runs/rfdetr/rfdetr_base_run1/results.json'
CHECKPOINT_BEST = 'runs/rfdetr/rfdetr_base_run1/checkpoint_best_total.pth'
TEST_IMGS_DIR   = Path('../data/test/images')

# ── Accuracy metrics ──────────────────────────────────────────────────────────
with open(RESULTS_JSON) as f:
    rf_results = json.load(f)

CLASSES = ['Sunglasses', 'Keys', 'Wallet']
test_entries = {e['class']: e for e in rf_results['class_map']['test']}

rfdetr_map50     = test_entries['all']['map@50']
rfdetr_map50_95  = test_entries['all']['map@50:95']
rfdetr_precision = test_entries['all']['precision']
rfdetr_recall    = test_entries['all']['recall']
rfdetr_per_class = {cls: test_entries[cls]['map@50'] for cls in CLASSES}

print('RF-DETR Base — test set results (from RF-DETR internal evaluator)')
print(f'  mAP50:     {rfdetr_map50:.4f}')
print(f'  mAP50-95:  {rfdetr_map50_95:.4f}')
print(f'  Precision: {rfdetr_precision:.4f}')
print(f'  Recall:    {rfdetr_recall:.4f}')
print('\n  Per-class mAP50:')
for cls in CLASSES:
    print(f'    {cls}: {rfdetr_per_class[cls]:.4f}')

# ── Checkpoint file size ───────────────────────────────────────────────────────
rfdetr_size_mb = os.path.getsize(CHECKPOINT_BEST) / 1e6
print(f'\n  Checkpoint size (best_total.pth): {rfdetr_size_mb:.2f} MB')

# ── GPU inference speed ───────────────────────────────────────────────────────
timing_model = RFDETRBase()
timing_model.optimize_for_inference()

test_images = sorted(TEST_IMGS_DIR.glob('*.jpg'))[:23]
times_ms = []
for img_path in test_images:
    img = Image.open(img_path).convert('RGB')
    t0 = time.perf_counter()
    _ = timing_model.predict(img, threshold=0.5)
    times_ms.append((time.perf_counter() - t0) * 1000)

rfdetr_infer_ms = sum(times_ms[3:]) / len(times_ms[3:])   # exclude warm-up
print(f'  Inference speed (GPU): {rfdetr_infer_ms:.2f} ms/image  '
      f'(avg over {len(times_ms)-3} images, warm-up excluded)')

RF-DETR Base — test set results (from RF-DETR internal evaluator)
  mAP50:     0.8635
  mAP50-95:  0.6812
  Precision: 0.9251
  Recall:    0.8715

  Per-class mAP50:
    Sunglasses: 0.9713
    Keys: 0.7806
    Wallet: 0.8386

  Checkpoint size (best_total.pth): 127.65 MB
[2026-03-25 12:15:25] [INFO] rf-detr - File rf-detr-base.pth already exists with correct MD5 hash.
[2026-03-25 12:15:26] [INFO] rf-detr - Loading pretrain weights


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  Inference speed (GPU): 11.08 ms/image  (avg over 20 images, warm-up excluded)


## Model Comparison

All models are evaluated on the **same held-out test split** at image size 640x640. Seven models are compared across three YOLO architectures (two size tiers each) plus one transformer baseline:

- **YOLOv8n / YOLOv8s** — CNN-based, established baseline (2023)
- **YOLO11n / YOLO11s** — improved efficiency and accuracy over v8 (2024)
- **YOLO26n / YOLO26s** — NMS-free and DFL-free; optimised for edge/mobile (2025)
- **RF-DETR Base** — transformer-based detector; heavyweight reference

The best-performing YOLO variant will be selected as the YOLO representative in the final 3-architecture comparison.

> **Dataset imbalance note:** The test set is skewed — Wallet 55.8 %, Keys 35.5 %, Glasses 8.7 %. Overall mAP is therefore weighted towards Wallet performance. Per-class AP50 is the more informative metric for understanding class-level detection quality.

In [4]:
import os
import pandas as pd
from ultralytics import YOLO

CLASSES   = ['Sunglasses', 'Keys', 'Wallet']
DATA_YAML = '../data/data.yaml'

ultralytics_runs = {
    'YOLOv8n  (50ep)': 'runs/detect/yolov8n_run1/weights/best.pt',
    'YOLOv8s  (50ep)': 'runs/detect/yolov8s_run1/weights/best.pt',
    'YOLO11n  (50ep)': 'runs/detect/yolo11n_run1/weights/best.pt',
    'YOLO11s  (50ep)': 'runs/detect/yolo11s_run1/weights/best.pt',
    'YOLO26n  (50ep)': 'runs/detect/yolo26n_run1/weights/best.pt',
    'YOLO26s  (50ep)': 'runs/detect/yolo26s_run1/weights/best.pt',
}

rows = []
for name, weights in ultralytics_runs.items():
    model = YOLO(weights)
    m     = model.val(data=DATA_YAML, split='test', verbose=False)
    size_mb = os.path.getsize(weights) / 1e6
    rows.append({
        'Model':          name,
        'Size (MB)':      round(size_mb,               2),
        'mAP50':          round(m.box.map50,            4),
        'mAP50-95':       round(m.box.map,              4),
        'Precision':      round(m.box.mp,               4),
        'Recall':         round(m.box.mr,               4),
        'AP50 Sunglasses': round(m.box.maps[0],         4),
        'AP50 Keys':      round(m.box.maps[1],          4),
        'AP50 Wallet':    round(m.box.maps[2],          4),
        'Infer GPU (ms)': round(m.speed['inference'],   2),
    })

# RF-DETR Base — populated from variables set in the RF-DETR evaluation cell above.
try:
    rows.append({
        'Model':          'RF-DETR Base (50ep)',
        'Size (MB)':      round(rfdetr_size_mb,                                    4),
        'mAP50':          round(rfdetr_map50,                                      4),
        'mAP50-95':       round(rfdetr_map50_95,                                   4),
        'Precision':      round(rfdetr_precision,                                  4),
        'Recall':         round(rfdetr_recall,                                     4),
        'AP50 Sunglasses': round(rfdetr_per_class.get('Sunglasses', float('nan')), 4),
        'AP50 Keys':      round(rfdetr_per_class.get('Keys',    float('nan')),     4),
        'AP50 Wallet':    round(rfdetr_per_class.get('Wallet',  float('nan')),     4),
        'Infer GPU (ms)': round(rfdetr_infer_ms,                                   2),
    })
except NameError:
    rows.append({k: None for k in ['Model', 'Size (MB)', 'mAP50', 'mAP50-95',
                                    'Precision', 'Recall', 'AP50 Sunglasses',
                                    'AP50 Keys', 'AP50 Wallet', 'Infer GPU (ms)']})
    rows[-1]['Model'] = 'RF-DETR Base (50ep)'
    print('Note: RF-DETR metrics not found — run the RF-DETR evaluation cell first.')

df = pd.DataFrame(rows).set_index('Model')
print(df.to_string())

Ultralytics 8.4.14 🚀 Python-3.10.19 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 492.7±612.7 MB/s, size: 71.0 KB)
val: Scanning /home/bradtab/git/FYP/ml-training/data/test/labels... 186 images, 10 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 186/186 1.2Kit/s 0.2s0.2s
val: New cache created: /home/bradtab/git/FYP/ml-training/data/test/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 19, len(boxes) = 219. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 9.7it/s 1.2s0.2s
                   all        186        219      0.919      0.923      0.931       0.74
Speed: 0.9ms prep

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics_to_plot = ['mAP50', 'AP50 Sunglasses', 'AP50 Keys', 'AP50 Wallet']
df_plot = df[metrics_to_plot].dropna(how='all')

x     = np.arange(len(df_plot.index))
width = 0.18

fig, ax = plt.subplots(figsize=(13, 5))
for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + i * width, df_plot[metric].fillna(0), width, label=metric)

ax.set_ylabel('mAP50')
ax.set_title('Model Comparison — mAP50 on Test Set')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(df_plot.index, rotation=15, ha='right')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('runs/comparison_map50.png', dpi=150)
plt.show()
print('Chart saved to runs/comparison_map50.png')

## TFLite Export & Mobile Model Sizes

Each Ultralytics model is exported in two formats for Android deployment:

- **FP32** — full-precision baseline; largest file, highest fidelity
- **INT8** — post-training quantisation calibrated on the training set; smallest file, fastest on devices with a fixed-point DSP/NPU (e.g. Snapdragon Hexagon)

> **RF-DETR note:** RF-DETR does not have an official TFLite export path through the Ultralytics API. A separate ONNX → TFLite conversion via `ai-edge-torch` or `onnx-tf` would be required and is left as future work.

> **YOLO26 advantage:** The removal of DFL and NMS from YOLO26 produces a simpler, hardware-friendly computation graph with fewer unsupported ops — resulting in a cleaner TFLite export and better INT8 quantisation quality.

In [ ]:
import os
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

DATA_YAML = '../data/data.yaml'

export_targets = {
    'YOLOv8n': 'runs/detect/yolov8n_run1/weights/best.pt',
    'YOLOv8s': 'runs/detect/yolov8s_run1/weights/best.pt',
    'YOLO11n': 'runs/detect/yolo11n_run1/weights/best.pt',
    'YOLO11s': 'runs/detect/yolo11s_run1/weights/best.pt',
    'YOLO26n': 'runs/detect/yolo26n_run1/weights/best.pt',
    'YOLO26s': 'runs/detect/yolo26s_run1/weights/best.pt',
}

size_rows = []

for name, weights in export_targets.items():
    model = YOLO(weights)
    pt_mb = os.path.getsize(weights) / 1e6

    # FP32 TFLite
    fp32_out   = model.export(format='tflite', imgsz=640)
    fp32_dir   = Path(fp32_out).parent
    fp32_mb    = sum(f.stat().st_size for f in fp32_dir.glob('*.tflite')
                     if 'int8' not in f.name) / 1e6

    # INT8 TFLite — calibrated on training images
    int8_out   = model.export(format='tflite', imgsz=640, int8=True, data=DATA_YAML)
    int8_dir   = Path(int8_out).parent
    int8_mb    = sum(f.stat().st_size for f in int8_dir.glob('*int8*.tflite')) / 1e6

    size_rows.append({
        'Model':            name,
        'PyTorch .pt (MB)': round(pt_mb,   2),
        'TFLite FP32 (MB)': round(fp32_mb, 2),
        'TFLite INT8 (MB)': round(int8_mb, 2),
    })
    print(f'{name}: pt={pt_mb:.2f} MB  |  fp32={fp32_mb:.2f} MB  |  int8={int8_mb:.2f} MB')

df_sizes = pd.DataFrame(size_rows).set_index('Model')
print('\n', df_sizes.to_string())